# Evolução temporal dos hits (2010–2026)

Uma coisa é saber que hits tendem a ter mais energia. Outra é saber se isso sempre foi assim ou se mudou. Se o perfil sonoro do que estoura está se deslocando, a gravadora precisa saber pra onde apostar em 2026 — não em 2015.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

df = pd.read_csv("../data/processed/tracks_clean.csv")

FEATURES = [
    "danceability", "energy", "valence", "tempo",
    "loudness", "speechiness", "acousticness",
    "instrumentalness", "duration_ms",
]

hits = df[df["is_hit"] == 1].copy()
print(f"Total de hits no dataset: {len(hits)}")
print(f"Período: {hits['year'].min()} a {hits['year'].max()}")

## Quantidade de hits por ano

Antes de olhar as features, preciso saber se tenho dados suficientes em cada ano. Se um ano tem 5 hits e outro tem 500, a média daquele ano com 5 não diz muita coisa.

In [ ]:
hits_per_year = hits.groupby("year").size().reset_index(name="count")

fig = px.bar(
    hits_per_year, x="year", y="count",
    title="Quantidade de hits por ano no dataset",
    labels={"year": "Ano", "count": "Hits"},
    text="count",
)
fig.update_layout(height=400, template="plotly_white")
fig.show()

## Média anual de cada feature (somente hits)

Filtro só os hits e calculo a média por ano. O resultado mostra como o perfil sonoro de um hit foi mudando. Se danceability subiu de 0.55 em 2010 pra 0.70 em 2024, isso conta uma história.

In [ ]:
yearly = hits.groupby("year")[FEATURES].mean().reset_index()
yearly.round(3)

In [ ]:
for feat in FEATURES:
    fig = px.line(
        yearly, x="year", y=feat,
        markers=True,
        title=f"{feat} — média anual dos hits",
        labels={"year": "Ano", feat: feat},
    )
    fig.update_traces(
        hovertemplate="Ano: %{x}<br>Valor: %{y:.3f}<extra></extra>",
        line_color="#636EFA",
    )

    vals = yearly[feat]
    peak_idx = vals.idxmax()
    valley_idx = vals.idxmin()

    fig.add_annotation(
        x=yearly.loc[peak_idx, "year"], y=vals[peak_idx],
        text=f"Pico: {vals[peak_idx]:.3f}",
        showarrow=True, arrowhead=2, ay=-30,
    )
    fig.add_annotation(
        x=yearly.loc[valley_idx, "year"], y=vals[valley_idx],
        text=f"Vale: {vals[valley_idx]:.3f}",
        showarrow=True, arrowhead=2, ay=30,
    )

    fig.update_layout(height=400, template="plotly_white")
    fig.show()

## Comparação direta: início vs fim do período

Pra deixar a mudança mais visível, comparo a média dos hits de 2010–2013 com a de 2022–2026. Duas épocas, lado a lado.

In [ ]:
early = hits[hits["year"].between(2010, 2013)][FEATURES].mean()
recent = hits[hits["year"].between(2022, 2026)][FEATURES].mean()

comparison = pd.DataFrame({
    "2010–2013": early,
    "2022–2026": recent,
    "variação": recent - early,
}).round(4)

comparison

In [ ]:
radar_feats = [f for f in FEATURES if f != "duration_ms"]

fig = go.Figure()
fig.add_trace(go.Scatterpolar(
    r=[early[f] for f in radar_feats],
    theta=radar_feats, fill="toself",
    name="2010–2013", opacity=0.5,
))
fig.add_trace(go.Scatterpolar(
    r=[recent[f] for f in radar_feats],
    theta=radar_feats, fill="toself",
    name="2022–2026", opacity=0.5,
))
fig.update_layout(
    title="Perfil sonoro dos hits: 2010–2013 vs 2022–2026",
    polar=dict(radialaxis=dict(visible=True)),
    height=500, template="plotly_white",
)
fig.show()

## O que mudou nos hits de 2010 a 2026

**O que subiu:**

Speechiness e danceability subiram ao longo dos anos. Isso bate com o que aconteceu na indústria: hip-hop e pop dançante dominaram os charts na segunda metade da década de 2010 e continuam fortes. Hits recentes falam mais (mais letra, mais rap, menos instrumental) e são mais feitos pra dançar do que os de 2010.

**O que caiu:**

Acousticness e instrumentalness caíram. Hits de 2010 ainda tinham espaço pra violão e piano solo. Hoje, a produção é quase toda eletrônica. Músicas acústicas e instrumentais continuam existindo, mas não no topo dos charts.

**O que isso conta pra gravadora:**

Se a gravadora vai investir num primeiro single em 2026, os números dizem que o caminho mais seguro é uma faixa dançável, com produção densa, vocal presente e duração curta. Não é garantia de hit — o TikTok e o acaso ainda existem — mas é o perfil que tem mais chance de entrar em playlists de streaming, e playlists são o principal canal de descoberta hoje.

Isso não quer dizer que uma balada acústica não pode estourar. Quer dizer que, na média, ela está nadando contra a corrente.